In [16]:
import requests
r = requests.get("https://www.ebi.ac.uk/chembl/api/data/status.json", timeout=30)
print(r.status_code)
print(r.json() if r.ok else r.text[:200])

200
{'activities': 24527044, 'chembl_db_version': 'ChEMBL_37', 'chembl_release_date': '2026-05-01', 'compound_records': 3824604, 'disinct_compounds': 2921148, 'publications': 101100, 'status': 'UP', 'targets': 18552}


In [17]:
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated.*")
import requests
import pandas as pd
from chembl_webresource_client.new_client import new_client

In [18]:
molecule  = new_client.molecule
mechanism = new_client.mechanism
activity  = new_client.activity
target    = new_client.target

In [20]:
# ============================================================================
# Shared hop: target_chembl_id -> UniProt accessions (with metadata retained)
# ============================================================================
import time
from chembl_webresource_client.http_errors import BaseHttpException

def _fetch_with_retries(qs, retries=3, base_sleep=1.5):
    """Fetch a ChEMBL queryset with retry/backoff for transient API/server failures."""
    last_err = None
    for attempt in range(retries):
        try:
            return list(qs)
        except BaseHttpException as err:
            last_err = err
            # ChEMBL occasionally returns transient HTML error pages; retry with backoff.
            if attempt < retries - 1:
                time.sleep(base_sleep * (2 ** attempt))
                continue
            raise
    if last_err is not None:
        raise last_err
    return []

def targets_to_uniprot(target_ids, human_only=True, single_protein_only=False):
    """{target_chembl_id: {'target_type','organism','accessions': set}}."""
    ids, out, chunk = sorted({t for t in target_ids if t}), {}, 50
    for i in range(0, len(ids), chunk):
        recs_qs = target.filter(target_chembl_id__in=ids[i:i+chunk]).only(
            ["target_chembl_id", "target_type", "organism", "target_components"])
        recs = _fetch_with_retries(recs_qs)
        for t in recs:
            ttype, org = t.get("target_type"), t.get("organism")
            keep = ((not human_only or org == "Homo sapiens")
                    and (not single_protein_only or ttype == "SINGLE PROTEIN"))
            accs = ({c["accession"] for c in (t.get("target_components") or [])
                     if c.get("accession")} if keep else set())
            out[t["target_chembl_id"]] = {"target_type": ttype, "organism": org,
                                          "accessions": accs}
    for tid in ids:
        out.setdefault(tid, {"target_type": None, "organism": None, "accessions": set()})
    return out

# ============================================================================
# Step 1 — resolve drug name -> parent ChEMBL id
# ============================================================================
def resolve_parent(drug_name):
    hits_qs = molecule.filter(molecule_synonyms__molecule_synonym__iexact=drug_name).only(
        ["molecule_chembl_id", "pref_name"]
    )
    hits = _fetch_with_retries(hits_qs)
    if not hits:
        raise ValueError(f"No ChEMBL match for {drug_name!r}")
    return hits[0]["molecule_chembl_id"], hits[0]["pref_name"]

# ============================================================================
# Step 2 — T_mech : curated mechanism targets (keep complexes/families)
# ============================================================================
def extract_T_mech(parent, human_only=True):
    rows_qs = mechanism.filter(parent_molecule_chembl_id=parent).only(
        ["target_chembl_id", "action_type", "mechanism_of_action",
         "direct_interaction", "molecular_mechanism"]
    )
    rows = _fetch_with_retries(rows_qs)
    rows = [m for m in rows if m.get("target_chembl_id")
            and m.get("molecular_mechanism") and m.get("direct_interaction")]
    tmap = targets_to_uniprot([m["target_chembl_id"] for m in rows],
                              human_only=human_only, single_protein_only=False)
    T = set().union(*(v["accessions"] for v in tmap.values())) if tmap else set()
    return T, rows

# ============================================================================
# Step 3 — T_bio : bioactivity targets (human, binding, exact relation, >= theta)
# ============================================================================
def extract_T_bio(parent, theta=6.0, human_only=True):
    rows_qs = activity.filter(parent_molecule_chembl_id=parent,
                              pchembl_value__isnull=False).only(
        ["target_chembl_id", "target_organism", "pchembl_value",
         "standard_type", "standard_relation", "assay_type"]
    )
    try:
        rows = _fetch_with_retries(rows_qs)
    except BaseHttpException as err:
        # Keep pipeline running if a single drug hits transient/unrecoverable API issues.
        print(f"Warning: activity query failed for {parent}: {err}")
        return set(), {}

    # qualify rows: binding assay, exact measurement, potency >= theta
    best = {}                                  # target_chembl_id -> max pChEMBL
    for a in rows:
        tid = a.get("target_chembl_id")
        if not tid:                                   continue
        if a.get("assay_type") != "B":                continue   # binding only
        if a.get("standard_relation") != "=":         continue   # drop censored (>,<)
        if human_only and a.get("target_organism") != "Homo sapiens": continue
        pc = float(a.get("pchembl_value") or 0)
        if pc < theta:                                continue
        best[tid] = max(best.get(tid, 0), pc)         # dedupe per target, keep strongest
    tmap = targets_to_uniprot(list(best),
                              human_only=human_only, single_protein_only=True)
    T = set().union(*(v["accessions"] for v in tmap.values())) if tmap else set()
    return T, best

# ============================================================================
# Step 4 — assemble the target profile record
# ============================================================================
def target_profile(drug_name, theta=6.0):
    parent, pref = resolve_parent(drug_name)
    T_mech, mech_rows = extract_T_mech(parent)
    T_bio,  bio_best  = extract_T_bio(parent, theta=theta)
    return {
        "drug": drug_name,
        "parent_chembl_id": parent,
        "pref_name": pref,
        "T_mech": sorted(T_mech),               # list of UniProt IDs
        "T_bio":  sorted(T_bio),                # list of UniProt IDs
        "T_union": sorted(T_mech | T_bio),
        "_mech_rows": mech_rows,                # audit trail
        "_bio_best": bio_best,
    }

In [5]:
Bivalirudin
Goserelin
Gramicidin D
Desmopressin
Cetrorelix
Daptomycin
Cyclosporine
Pyridoxal phosphate
Cyanocobalamin
Histidine

SyntaxError: invalid syntax (647739460.py, line 3)

In [21]:
# Extract some of the drugs from the full list
import pandas as pd
df = pd.read_csv(r"C:\Users\ashto\ddi-prediction\data\raw\drugbank_approved_small_2369_drugs.csv")
df_drug_names = df['drug_name']
df_drug_names.head(10)

0            Bivalirudin
1              Goserelin
2           Gramicidin D
3           Desmopressin
4             Cetrorelix
5             Daptomycin
6           Cyclosporine
7    Pyridoxal phosphate
8         Cyanocobalamin
9              Histidine
Name: drug_name, dtype: object

In [22]:
# Create the new df that will append the new targets

chembl_target_profiles_list = []

In [23]:
import asyncio


def _profile_to_row(profile):
    return {
        'Drug': profile['drug'],
        'T_mech': profile['T_mech'],
        'T_bio': profile['T_bio'],
        'T_union': profile['T_union'],
    }


async def _process_drug_async(drug, theta, sem):
    drug = str(drug)
    try:
        async with sem:
            profile = await asyncio.to_thread(target_profile, drug, theta)
        return {'ok': True, 'profile': profile, 'error': None, 'drug': drug}
    except ValueError as e:
        return {'ok': False, 'profile': None, 'error': str(e), 'drug': drug}
    except Exception as e:
        # Keep long runs alive when the remote API intermittently fails.
        return {'ok': False, 'profile': None, 'error': f"Skipping {drug!r} due to API/runtime error: {e}", 'drug': drug}


async def collect_profiles_async(drug_names, theta=6.0, max_concurrency=8, progress_every=25):
    sem = asyncio.Semaphore(max_concurrency)
    tasks = [
        asyncio.create_task(_process_drug_async(drug, theta, sem))
        for drug in drug_names
        if pd.notna(drug)
    ]

    rows = []
    total = len(tasks)
    done = 0

    for fut in asyncio.as_completed(tasks):
        result = await fut
        done += 1

        if result['ok']:
            profile = result['profile']
            rows.append(_profile_to_row(profile))
        else:
            print(result['error'])

        if done % progress_every == 0 or done == total:
            print(f"Processed {done}/{total} drugs")

    return pd.DataFrame(rows)


chembl_target_profiles = await collect_profiles_async(
    df_drug_names,
    theta=6.0,
    max_concurrency=8,
    progress_every=25,
)
chembl_target_profiles.head()

No ChEMBL match for 'Choline'
Processed 25/2369 drugs
No ChEMBL match for 'NADH'
No ChEMBL match for 'Lipoic acid'
Processed 50/2369 drugs
Processed 75/2369 drugs
No ChEMBL match for 'Bethanidine'
Processed 100/2369 drugs
Processed 125/2369 drugs
Processed 150/2369 drugs
Processed 175/2369 drugs
Processed 200/2369 drugs
Processed 225/2369 drugs
Processed 250/2369 drugs
Processed 275/2369 drugs
Processed 300/2369 drugs
Processed 325/2369 drugs
Processed 350/2369 drugs
Processed 375/2369 drugs
Processed 400/2369 drugs
Processed 425/2369 drugs
Processed 450/2369 drugs
No ChEMBL match for 'Pentosan polysulfate'
Processed 475/2369 drugs
Processed 500/2369 drugs
Processed 525/2369 drugs
No ChEMBL match for 'Clidinium'
Processed 550/2369 drugs
No ChEMBL match for 'Cryptenamine'
No ChEMBL match for 'Dicyclomine'
Processed 575/2369 drugs
Processed 600/2369 drugs
No ChEMBL match for 'Benzphetamine'
Processed 625/2369 drugs
No ChEMBL match for 'Benzylpenicilloyl polylysine'
Processed 650/2369 dru

,Drug,T_mech,T_bio,T_union
0,Daptomycin,[],[],[]
1,Bivalirudin,[P00734],[],[P00734]
2,Pyridoxal phosphate,[],[],[]
3,Goserelin,[P30968],[],[P30968]
4,Cetrorelix,[P30968],[P30968],[P30968]


In [24]:
# Send the target profiles to a CSV file for later 

chembl_target_profiles.to_csv(r"C:\Users\ashto\ddi-prediction\data\sample\chembl_target_profiles.csv", index=False)



In [26]:
# Merge dataframes

merged_df = pd.merge(df, chembl_target_profiles,  left_on='drug_name', right_on='Drug', how='left')

In [29]:
merged_df.to_csv('merged_df_drugbank_approved_small_2369_drugs_with_chembl_target_by_name.csv', index=False)

In [ ]:
# ============================================================================
# Run it for Bivalirudin and Goserelin
# ============================================================================
profile = target_profile(, theta=6.0)

print(f"{profile['drug']}  ->  {profile['parent_chembl_id']}  ({profile['pref_name']})")
print(f"  T_mech ({len(profile['T_mech'])}): {profile['T_mech']}")
print(f"  T_bio  ({len(profile['T_bio'])}): {profile['T_bio']}")
print(f"  T(d)   ({len(profile['T_union'])}): {profile['T_union']}")

print("\n  mechanism audit:")
for m in profile["_mech_rows"]:
    print(f"    {m['target_chembl_id']}  {m.get('action_type')}  |  {m.get('mechanism_of_action')}")

Bivalirudin  ->  CHEMBL5314348  (BIVALIRUDIN)
  T_mech (1): ['P00734']
  T_bio  (0): []
  T(d)   (1): ['P00734']

  mechanism audit:
    CHEMBL204  INHIBITOR  |  Thrombin inhibitor


In [6]:
chembl_df = pd.read_csv(f"C:\\Users\\ashto\\ddi-prediction\\data\\sample\\chembl_target_profiles.csv")

whole_df = pd.read_csv(f"C:\\Users\\ashto\\ddi-prediction\\data\\raw\\drugbank_approved_small_2369_drugs.csv")
print(len(chembl_df))
print(len(whole_df))


2502
2369


In [14]:
set_chembl = set(chembl_df['Drug'].dropna())
set_whole = set(whole_df['drug_name'].dropna())
print(len(set_chembl))
print(len(set_whole))

2492
2369


In [10]:
missing_drugs = set(chembl_df['Drug']) - set(whole_df['drug_name'])

pd.DataFrame({'missing_drugs': list(missing_drugs)})

,missing_drugs
0,Amrinone
1,Alprenolol
2,Diethylstilbestrol
3,Ibuproxam
4,Isothipendyl
...,...
917,Dexchlorpheniramine
918,Fenpiverinium
919,Dopexamine
920,Iproclozide


In [13]:
col1_clean = chembl_df['Drug'].dropna()
missing_df = col1_clean[~col1_clean.isin(whole_df['drug_name'])]

print(missing_df)

26            Tryptophan
38            Masoprocol
46               Esmolol
48         Ethchlorvynol
51            Vidarabine
              ...       
2470    Dermatan sulfate
2471        Sulfametrole
2474       Levosulpiride
2484        Mobocertinib
2489          Tegoprazan
Name: Drug, Length: 922, dtype: object


In [23]:
test = chembl_df['Drug'].unique()
print(len(test))
print(test)

2492
['Bivalirudin' 'Goserelin' 'Gramicidin D' ... 'Lomifylline' 'Sebetralstat'
 'Vanzacaftor']
